# Exercise 0 - PydanticAI and OpenRouter

## 0. Summarize job ads
---
---
In the data folder, there are a few job ads taken from arbetsförmedlingen.se.

a) Read all the jobs ads into python

b) Create a function that uses PydanticAI agent to summarize a job ad. This function should take in an ad as its parameter and return a summary.

c) Now create and export markdown files for each job ad and its corresponding summary.

d) Try to take other job ads from arbetsförmedlingen.se and see how well your function performs.

e) Make the output structured using a pydantic model together with PydanticAI agent. The output from the agent should have the following fields:

- job_title
- description
- summary
- responsibilities
- words_in_article

In [ ]:
# a)
ads = {path.stem: path.read_text(encoding="utf-8")
       for path in sorted(DATA_DIR.glob("ads*.txt"))}
print(ads.keys())

dict_keys(['ads1', 'ads2', 'ads3'])


In [9]:
import os
from dotenv import load_dotenv
from pydantic_ai import Agent
from pydantic_ai.models.openai import OpenAIChatModel
from pydantic_ai.providers.openai import OpenAIProvider

load_dotenv()  # expects NVIDIA_API_KEY in your .env file

MODEL = OpenAIChatModel(
    "nvidia/nemotron-3.5-super-vl-120b-a12b",
    provider=OpenAIProvider(
        base_url="https://integrate.api.nvidia.com/v1",
        api_key=os.getenv("NVIDIA_API_KEY"),
    ),
)

In [ ]:
# b)
summary_agent = Agent(
    MODEL,
    system_prompt=(
        "You summarize Swedish job ads. Write a concise summary in English "
        "(3–5 sentences) covering the role, employer, key requirements "
        "and location. Do not invent details that are not in the ad."
    ),
)

async def summarize_ad(ad: str) -> str:
    result = await summary_agent.run(ad)
    return result.output

summary = await summarize_ad(ads["ads1"])
print(summary)

Instabee, a European last-mile delivery and e-commerce company formed from the merger of Budbee and Instabox, is hiring a Data Engineer for its newly formed Data Platform team. The role involves building and maintaining a modern data stack on GCP and Snowflake, integrating sources such as APIs, MongoDB, and MySQL, orchestrating pipelines with Airflow, and modeling data with DBT, SQL, and Python. Candidates should have experience with CI/CD, infrastructure as code, and data platform design patterns, along with a university degree in a technical or analytical field. The position is based in Stockholm with a hybrid arrangement requiring three office days per week.


In [ ]:
# c)
OUTPUT_DIR = Path("summaries")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

for name, ad in ads.items():
    summary = await summarize_ad(ad)
    markdown = f"# {name}\n\n## Summary\n\n{summary}\n\n## Original ad\n\n{ad}\n"
    (OUTPUT_DIR / f"{name}.md").write_text(markdown, encoding="utf-8")
    print(f"Saved {name}.md")

Saved ads1.md
Saved ads2.md
Saved ads3.md


In [ ]:
# d)

In [ ]:
# e) structured output.
from pydantic import BaseModel, Field

class JobAdSummary(BaseModel):
    job_title: str = Field(description="The job title as stated in the ad")
    description: str = Field(description="Short description of the employer and role")
    summary: str = Field(description="Concise 3–5 sentence summary of the ad")
    responsibilities: list[str] = Field(description="Main tasks and responsibilities, one per item")
    words_in_article: int = Field(ge=0, description="Number of words in the original ad")

structured_agent = Agent(
    MODEL,
    output_type=JobAdSummary,
    system_prompt=(
        "You extract structured information from Swedish job ads. "
        "Answer in English and only use information found in the ad."
    ),
)

async def summarize_ad_structured(ad: str) -> JobAdSummary:
    result = await structured_agent.run(ad)
    # LLMs are bad at counting, so overwrite with the real word count
    return result.output.model_copy(update={"words_in_article": len(ad.split())})

for name, ad in ads.items():
    job = await summarize_ad_structured(ad)
    print(f"--- {name} ---")
    print(job.model_dump_json(indent=2))

--- ads1 ---
{
  "job_title": "Data Engineer",
  "description": "Instabee (Instabox/Budbee), a European last-mile delivery and e-commerce enablement company, is hiring a Data Engineer to join its newly formed Data Platform team in Stockholm. The role involves shaping a modern data platform built on GCP, Snowflake, FiveTran, DBT, Python, and Tableau, combining Data Engineering with Analytics Engineering responsibilities.",
  "summary": "Instabee is seeking a Data Engineer with an interest in Analytics Engineering to join its newly formed Data Platform team. The team is building a modern data stack using FiveTran, Python, DBT, GCP, Snowflake, and Tableau to serve stakeholders across the organization. The role combines setting up source integrations and data platform infrastructure with data modeling and analytical project support. The position is based in Stockholm with hybrid work (3 days in the office).",
  "responsibilities": [
    "Lead the setup of integration patterns from sources 

CancelledError: 

## 4. Theory questions
___
---

#### a) Explain the primary purpose of Pydantic. What core problems does it solve for developers?
Pydantic is mainly used for data validation and data prasing in Python. It allows developers to define what structure and types data should have and then validates incoming data aginst those rules.

It solves problems such as:
- Incorrect data types
- Missing required fields 
- Invalid values 
- Unstructured input data 
- Manually writing lots of validation code

#### b) In Pydantic, what is the key difference between using model_validate() and model_validate_json()? When would you use each one?
The main diffrence is the format of the unout data.
- model_validate() is used when you already have Python data, such as a dictionary
- model_validate_json() is used when the data is in JSON format, usually as a JSN string or bytes. 

#### c) How can you define a field in a Pydantic model that has a specific value constraint, such as a minimum or maximum number? Give an example.
Use Field() with constraint atguments: gt/ge (greater than / greater or equal), lt/le (less than / less or equal), min_length / max_length for stings and lists, and pattern for regex. 

#### d) Describe how Pydantic plays a role in PydanticAI agents
Ptydantic is what turns free-text LLM output into reliable, typed data. 
- Output schema: you pass a Pydantic model as the agent´s output_type. PydanticAI converts it to a JSON schema and gives it to the LLM, usually as a tool or function defenition, so the model knows exactly which fields to return.
- Validation: the LLM´s response is validated against the model. If this fails, for example because a field is missing or has the wrong type, PydanticAI sends the error back to the LLM and asks it to retry. 
- Typed results: results.output is a real model instance, so you get attribute access and editor autocompletion. 
- Tools and dependecies: tool function argumnets are also validated with Pydantic, base on their type hints.

#### e) What is a computed field in Pydantic?
A computed field is a vlaue derived from other fields rather than supplied as input. You define it with the @computed_field decorator on a property. It isn´t vlaidated from input, but it appears in model_dump(), JOSn Schema.

#### f) What are tokens in LLMs?
Tokens are the units of text an LLM actually reads and writes. A tokenizer splits text into peices that are often whole common words, parts of words, punctuation or spaces. Each token maps to an ID in the model´s vocabulary. 
* In English, a rough rule is that 1 token i about 4 characters, or about 3/4 of a word. In swedish and other languages often need more tokens per word. 

#### g) What are few shot examples?
Few-shot examples are a small number of inout -> output demonstrations you put in the prompt to show the model what you want. The model infers the pattern, format and standards from them, without any retraining.

````
Classify the sentiment.

Review: "Arrived broken, terrible support." → negative
Review: "Does the job, nothing special." → neutral
Review: "Absolutely love it!" → positive

Review: "Battery died after two days." →
````

## Glossary
---
---
Fill in this table either by copying this into your own markdown file or copy it into a spreadsheet if you feel that is easier to work with.

| terminology        | explanation |
| ------------------ | ----------- |
| few shot learning  |Showing an LLM a few input -> output exmaples in the prompt so it learns the desired pattern ir format, without any retraining.             |
| token              |THe basic unit of text an LLM proccesses: a word, a punbctuation mark or a space. Cost and context limits are measured in tokens.         |
| context window     |The maximum number of tokens a model can hanlde at once, counting the prompt, conversation history and generated output together. Anything beyond it is cut off or rejected.              |
| pydantic           |A Pyrhon library for data validation and prasing based on on type hints. It checks and converts data at runtime and serializes models to dicts and JSON.             |
| BaseModel          |The base class you inherit from to define a Pydantic model. Fields are declared as type-annotated class attributes and validated when an instance is created.             |
| Field              |A function for adding extra information to a model field: default vlaues, constraints (gt, le, min_length, etc). descriptions and aliases.             |
| model_fields       |A class attribute holding a dict of a model´s field names and their FieldInfo (type, default, constraints). Used to inspect a model´s structure.             |
| computed_field     |A decorator that turns a property into a field derived from other fields. It isn´t provided as input, but it is included in model_dump(), JSON output and the Schema.             |
| model_validate     |A class method that validates Python object (usually a dict) and returns a model instance. It raises ValidationError if the data is invalid.             |
| model_dump         |An instance method that converts a model into a plain Python dict. model_dump_json() gives a JSON string instead.              |
| type hinting       |Annotating variables, parameters and return values with their expected types e.g. def f(x: int) -> str. Python itself dosen´t enforce hints: Pydantic uses them to validate at runtime.              |
| serialization      |Converting an in-memory object into a storable or transformable format, such as a dict or JSON string. In Pydantic: model_dump() / model_dump_json().             |
| deserialization    |Turning stored or recived data, such as JSON, back into a Python object. In Pydantic: model_validate() / model_dump_json().             |
| prompt engineering |Daesigning and refining the instructions, context and examples given to an LLM to get more accurate, consistent and useful output.             |
| zero-shot learning |Asknig an LLM to perform a task with only instructions and no examples, relying on what it learned in pretraining.             |
| OpenRouter         |A service that gives access to many LLMs (OpenAI, Antropic, Google, Meta, etc.) through one OpenAI-compatible API and on API key, with pay-per-token pricing.             |
| Agent              |In PydanticAI, the main class that wraps an LLM together with a system prompt, tools, dependencies and an output type. You call it to run a task.             |
| system prompt      |Instructions given to the model before the user’s input, setting its role, behavior, rules and output style for the whole conversation.             |
| agent.run          |The async method that sends a prompt to the agent and returns a result whose .output holds the validated response. agent.run_sync() is the synchronous version.             |
| structured ouput | LLM output that follows a defined schema, such as a Pydantic model, instead of free text, so it can be validated and used directly in code. |